# 3.7 能同時找不同關聯嗎？


讀「小貓追著拿球的孩子」時，你可能一方面找誰在追，一方面找球屬於誰。若只用一種匹配與混合方式，所有資訊都擠進同一份讀取結果。注意力可以同時建立幾種分別計算的讀法，各有自己的Q、K、V與讀取比例，再把取回的向量拼接。每種讀法叫一個head（注意力頭），整個機制叫多頭注意力（multi-head attention）。

「不同讀法」是可學習的自由度，不是作者預先把第一個head命名為文法、第二個命名為物體。要觀察訓練後是否分工，必須看實際行為，不能光看圖上的不同顏色就下結論。每個head仍按 [3.4](https://birdhackor.github.io/tiny-perceptron-vlm/3.4.html) 匹配與混合，猜下一字時也都要遵守 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的因果許可。

假設整個位置表示有D=8個特徵，使用H=2個head，通常讓每個head用D/H=4個特徵。先回顧 [2.3](https://birdhackor.github.io/tiny-perceptron-vlm/2.3.html)：Linear（線性層）用各輸入乘上可調權重再相加，形成新特徵，也可另加可調的偏移量。這裡把用線性配方重新組合特徵的操作叫投影。X先用三套各自的配方形成Q、K、V，每份在每個位置都有八數；分頭發生在投影之後。

以一筆四位置的Q為例，原形狀 `[筆數,位置,特徵]=[1,4,8]`。先把最後的八格分成兩組四格，得到 `[筆數,位置,頭,每頭特徵]=[1,4,2,4]`。若某位置的八數是 `[1,2,3,4,5,6,7,8]`，head0取 `[1,2,3,4]`，head1取 `[5,6,7,8]`；四個位置都各自這樣分，沒有把位置分給不同頭。再交換「位置」與「頭」兩個軸，得到 `[筆數,頭,位置,每頭特徵]=[1,2,4,4]`，讓每個head各自處理整串四位置。K、V也各自這樣分組。各head輸出四特徵，最後按head順序拼接回每位置八特徵，再經過輸出配方混合；拼接的背景見 [2.2](https://birdhackor.github.io/tiny-perceptron-vlm/2.2.html)。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
import torch
from tiny_perceptron.attention import CausalAttention

torch.manual_seed(42)
layer = CausalAttention(width=8, heads=2)
x = torch.randn(1, 4, 8)
out, cache = layer(x)
print("輸出", out.shape)
print("保存的K", cache[0].shape)
assert out.shape == x.shape

`torch.manual_seed(42)`固定PyTorch產生隨機數的起點，讓從頭重跑時的初始權重與輸入可重現。`torch.randn`產生隨機浮點數，這裡用來模擬位置特徵，只測分組形狀，不把它當成真實句子或已學好的表示。

輸入x是一筆、四位置、每位置八特徵，形狀 `[1,4,8]`。`CausalAttention` 是專案封裝好的因果多頭層，內部用Linear形成Q/K/V，分組後做注意力，再拼回。輸出仍 `[1,4,8]`，表示每位置的資訊被更新，而不是字數增加了或head變成另一筆句子。

這個工具還回傳一對保存的K、V，叫cache（快取），讓之後逐字生成有機會重用已算過的資料；此處只看其形狀，不需要使用快取生成。`cache[0]` 是K，形狀 `[1,2,4,4]`：一筆、兩head、四位置、每head四特徵。真正每head的讀取比例則是一張四乘四表；本節印的K不是那張權重表，要分清特徵軸與候選位置軸。

下圖兩條分支是head0、head1。分支中的「4位置×4特徵」表示每個head都讀整串，並不是第一個只讀前兩字、第二個只讀後兩字。「4×4比例表」的列是查詢位置、欄是可讀位置，與每位置四個特徵的4用途不同。向下的箭頭把兩份每位置四數的輸出拼成八數，才交給最後的Linear。兩種顏色只是分組標記，沒有指定它們要學哪種語意。


In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/foundations_heads.svg")))

若D固定、把H增加，每head的特徵數會減少，並不是每個head都繼續拿到完整D個新特徵。D必須能被H整除，否則八格無法平均分成三組，工具會報錯。這也是讀shape時要把「位置數」「head數」「每head特徵數」分別寫清楚的理由。

練習只把heads從2改成4，先預測輸出形狀保持 `[1,4,8]`，K改成 `[1,4,4,2]`，再執行核對。隨機初始數值可能改變，但不能憑這一次輸出說四head已比兩head聰明；這個實驗只驗證拆分與合併的接口，能力仍須訓練與評估。
